# Implementación Paralela con CUDA

Este cuaderno corresponde a los pasos 3, 4 y 5 del proyecto. Primero identifica, con base en las mediciones del cuaderno anterior, qué operaciones conviene paralelizar. Después describe el diseño paralelo y lo implementa en dos fases separadas.

1. **Preparación de datos en varios núcleos del CPU.** La lectura, el redimensionado y la normalización de imágenes se reparten entre varios procesos.
2. **Entrenamiento del modelo en la GPU.** TensorFlow ejecuta la matemática de la red neuronal con CUDA (la plataforma de cómputo paralelo de NVIDIA) y cuDNN (su librería de operaciones para redes neuronales).

Además, se incluye un experimento de contraste. Un kernel propio (una función que se ejecuta en paralelo sobre miles de hilos de la GPU), escrito con Numba, redimensiona las imágenes en la GPU. Así se evalúa si conviene mover también la preparación de datos a la GPU.

## Configuración del Entorno

En este cuaderno la GPU queda visible. La opción de crecimiento de memoria hace que TensorFlow reserve memoria de la GPU solo cuando la necesita, para que Numba también pueda usarla.

In [ ]:
import os
import sys
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from src import config
from src.data.make_dataset import class_names, make_dataset
from src.features.build_features import preprocess_sequential
from src.features.parallel_features import decode_parallel, make_pool, preprocess_parallel, split_chunks
from src.models.train_model import (build_cnn, device_name, enable_memory_growth, evaluate,
                                    host_to_device_seconds, set_seed, train)
from src.utils.monitor import ResourceMonitor
from src.utils.system_info import print_report
from src.utils.timing import Timer, save_times, summarize

enable_memory_growth()
print("Dispositivo de TensorFlow:", device_name())

In [ ]:
print_report(include_nvidia_smi=True)

## Identificación de Oportunidades de Paralelización

La siguiente celda recupera los tiempos de la solución secuencial y calcula qué porcentaje del tiempo total ocupa cada fase y cada etapa del preprocesamiento.

In [ ]:
seq_times = pd.read_csv(config.RESULTS_DIR / "sequential_times.csv")
seq_mean = seq_times[seq_times["note"] != "first"].groupby("phase", sort=False)["seconds"].mean()
share = pd.DataFrame({"segundos": seq_mean, "porcentaje": 100 * seq_mean / seq_mean.sum()})
display(share.round(2))

stages = pd.read_csv(config.RESULTS_DIR / "01_preprocess_stages.csv", index_col="etapa")
stages.round(3)

## Diseño del Algoritmo Paralelo

### Fase 1 en el CPU

Cada imagen se procesa de forma independiente, por lo que el problema es de paralelismo de datos (la misma operación se aplica a muchos datos distintos sin que unos dependan de otros). Las 14,500 rutas se dividen en bloques contiguos, cuatro bloques por proceso, para equilibrar la carga. Cada proceso lee y redimensiona su bloque, y devuelve un arreglo de enteros de 8 bits. El proceso principal une los bloques y normaliza todo el arreglo en una sola operación vectorizada (una operación que NumPy aplica a todo el arreglo a la vez).

Se usan procesos y no hilos porque el intérprete de Python tiene un candado global (GIL, Global Interpreter Lock) que impide que varios hilos ejecuten código Python al mismo tiempo.

### Fase 2 en la GPU

TensorFlow divide cada época en lotes de 64 imágenes. Cada lote se copia a la memoria de la GPU, y cuDNN lanza los kernels de convolución, activación y agrupamiento. En estos kernels miles de hilos calculan en paralelo los elementos de los mapas de activación. La distribución en hilos, bloques y grid la decide cuDNN de forma automática según el tamaño de cada capa.

### Experimento con un Kernel Propio

Para el redimensionado en la GPU se diseña un kernel en el que cada hilo calcula un píxel de salida con sus tres canales. Los hilos se agrupan en bloques de 16 × 16 (256 hilos), y cada bloque cubre una región de 16 × 16 píxeles de la imagen de salida. El grid tiene 4 × 4 bloques por imagen en los ejes x e y, y una capa por imagen en el eje z. La siguiente celda calcula estas dimensiones para un lote.

In [ ]:
from src.features.cuda_kernels import BLOCK, grid_dims

grid = grid_dims(config.GPU_CHUNK)
threads_per_block = BLOCK[0] * BLOCK[1] * BLOCK[2]
n_blocks = grid[0] * grid[1] * grid[2]
pd.Series({
    "Imágenes por lote": config.GPU_CHUNK,
    "Bloque (x, y, z)": BLOCK,
    "Grid (x, y, z)": grid,
    "Hilos por bloque": threads_per_block,
    "Bloques por lote": n_blocks,
    "Hilos por lote": n_blocks * threads_per_block,
}).to_frame("valor")

## Implementación y Medición

### Fase 1 con Varios Procesos del CPU

Primero se ejecuta la versión secuencial una vez, como referencia para comprobar que la versión paralela produce exactamente el mismo resultado. Esta ejecución también carga los archivos en la caché del sistema operativo, de modo que todas las mediciones parten de las mismas condiciones. Luego se mide el preprocesamiento con 1, 2, 4, 8, 16 y 32 procesos, con 5 repeticiones cada uno. Antes de las repeticiones se hace una ejecución de calentamiento, que no se promedia, para que los procesos ya estén iniciados.

In [ ]:
manifest = make_dataset()
classes = class_names(manifest)
paths = manifest["path"].tolist()

with Timer() as t_ref:
    X_ref = preprocess_sequential(paths)
print(f"Referencia secuencial: {t_ref.elapsed:.2f} s")

worker_counts = [n for n in config.WORKER_COUNTS if n <= os.cpu_count()]
print("Configuraciones de procesos:", worker_counts)

In [ ]:
cpu_records = []
cpu_usage_par = None

for n in worker_counts:
    with Timer() as t_start:
        pool = make_pool(n)
        X_par = preprocess_parallel(paths, pool, n)
    cpu_records.append({"phase": "preprocess", "workers": n, "rep": 0, "seconds": t_start.elapsed, "note": "warmup"})
    assert np.array_equal(X_par, X_ref), "El resultado paralelo difiere del secuencial"

    monitor = ResourceMonitor(interval=0.25)
    with monitor:
        for rep in range(1, config.N_REPS_PREPROCESS + 1):
            with Timer() as t:
                X_par = preprocess_parallel(paths, pool, n)
            cpu_records.append({"phase": "preprocess", "workers": n, "rep": rep, "seconds": t.elapsed, "note": ""})
    pool.close()
    pool.join()
    if n == worker_counts[-1]:
        cpu_usage_par = monitor.to_dataframe()

    reps = [r["seconds"] for r in cpu_records if r["workers"] == n and r["rep"] > 0]
    print(f"{n:>2} procesos: {np.mean(reps):.2f} s en promedio (calentamiento {t_start.elapsed:.2f} s)")

cpu_df = save_times(cpu_records, config.RESULTS_DIR / "02_cpu_parallel_times.csv")
cpu_usage_par.to_csv(config.RESULTS_DIR / "02_cpu_usage_parallel.csv", index=False)

In [ ]:
cpu_summary = pd.DataFrame({
    n: summarize(g.loc[g["rep"] > 0, "seconds"].tolist()) for n, g in cpu_df.groupby("workers")
}).T.rename_axis("procesos").round(3)
cpu_summary

### Experimento de Contraste con el Kernel de Numba

En esta variante el CPU solo lee y decodifica las imágenes (con todos los procesos), y la GPU realiza el redimensionado y la normalización. El trabajo se divide en lotes de 2,048 imágenes. Para cada lote se mide por separado la decodificación en el CPU, la copia de las imágenes originales de 200 × 200 a la GPU, la ejecución del kernel y la copia del resultado de vuelta al CPU. La primera llamada al kernel incluye su compilación, por lo que se mide aparte.

In [ ]:
from src.features.cuda_kernels import resize_normalize_gpu

n_workers = worker_counts[-1]
chunks = split_chunks(paths, int(np.ceil(len(paths) / config.GPU_CHUNK)))
pool = make_pool(n_workers)
sample = decode_parallel(chunks[0][:n_workers], pool, n_workers)

with Timer() as t_compile:
    resize_normalize_gpu(sample)
print(f"Compilación y primera llamada del kernel: {t_compile.elapsed:.2f} s")

numba_records = []
for rep in range(0, config.N_REPS_PREPROCESS + 1):
    totals = {"decode": 0.0, "h2d": 0.0, "kernel": 0.0, "d2h": 0.0}
    abs_diff_sum, abs_diff_max, start = 0.0, 0.0, 0
    for chunk in chunks:
        with Timer() as t_dec:
            raw = decode_parallel(chunk, pool, n_workers)
        X_chunk, times = resize_normalize_gpu(raw)
        totals["decode"] += t_dec.elapsed
        for key, value in times.items():
            totals[key] += value
        diff = np.abs(X_chunk - X_ref[start:start + len(chunk)])
        abs_diff_sum, abs_diff_max = abs_diff_sum + diff.sum(), max(abs_diff_max, diff.max())
        start += len(chunk)
    total = sum(totals.values())
    numba_records.append({"rep": rep, **totals, "total": total, "note": "warmup" if rep == 0 else ""})
    print(f"Repetición {rep}: total {total:.2f} s " + ", ".join(f"{k} {v:.3f} s" for k, v in totals.items()))

pool.close()
pool.join()

print(f"Diferencia con la versión de PIL: media {abs_diff_sum / X_ref.size:.4f}, máxima {abs_diff_max:.4f}")
print(f"Datos copiados a la GPU por repetición: {len(paths) * 200 * 200 * 3 / 1024**2:.0f} MB")

numba_df = save_times(numba_records, config.RESULTS_DIR / "02_numba_times.csv")
numba_df[numba_df["rep"] > 0].drop(columns=["rep", "note"]).agg(["mean", "std"]).round(4)

### Fase 2 con la GPU

El entrenamiento usa exactamente el mismo modelo, los mismos datos, la misma semilla, 10 épocas y lotes de 64 imágenes que la versión secuencial. Los datos de entrada son los que produjo la Fase 1. Dentro del tiempo de entrenamiento ya se incluyen las copias de cada lote del CPU a la GPU. Además, se mide aparte cuánto tarda copiar todo el conjunto de entrenamiento a la GPU. Durante el entrenamiento se registra el uso de la GPU y de su memoria con `nvidia-smi`.

In [ ]:
y_all = manifest["label_idx"].to_numpy()
masks = {s: (manifest["split"] == s).to_numpy() for s in ("train", "val", "test")}
X_train, y_train = X_par[masks["train"]], y_all[masks["train"]]
X_val, y_val = X_par[masks["val"]], y_all[masks["val"]]
X_test, y_test = X_par[masks["test"]], y_all[masks["test"]]
del X_ref

gpu_records, gpu_epochs = [], []
gpu_usage = None

for rep in range(1, config.N_REPS_TRAIN + 1):
    t_h2d = host_to_device_seconds(X_train)
    set_seed()
    model = build_cnn(n_classes=len(classes))
    with ResourceMonitor(interval=0.5, gpu=True) as mon:
        result = train(model, X_train, y_train, X_val, y_val)
    test_loss, test_acc, t_inf = evaluate(model, X_test, y_test)

    gpu_records += [
        {"phase": "transfer_h2d", "device": "GPU", "rep": rep, "seconds": t_h2d},
        {"phase": "train", "device": "GPU", "rep": rep, "seconds": result.total_time},
        {"phase": "inference", "device": "GPU", "rep": rep, "seconds": t_inf},
    ]
    for epoch, sec in enumerate(result.epoch_times, start=1):
        gpu_epochs.append({
            "device": "GPU", "rep": rep, "epoch": epoch, "seconds": sec,
            "train_acc": result.history["accuracy"][epoch - 1],
            "val_acc": result.history["val_accuracy"][epoch - 1],
        })
    if gpu_usage is None:
        gpu_usage = mon.to_dataframe()
    print(f"Repetición {rep}: entrenamiento {result.total_time:.1f} s, accuracy de prueba {test_acc:.4f}, "
          f"inferencia {t_inf:.2f} s, copia a la GPU {t_h2d:.3f} s")

gpu_df = save_times(gpu_records, config.RESULTS_DIR / "gpu_times.csv")
gpu_epochs_df = save_times(gpu_epochs, config.RESULTS_DIR / "gpu_epochs.csv")
gpu_usage.to_csv(config.RESULTS_DIR / "02_gpu_usage_train.csv", index=False)
print(f"Tamaño de X_train: {X_train.nbytes / 1024**2:.1f} MB")

In [ ]:
gpu_summary = pd.DataFrame({
    phase: summarize(g["seconds"].tolist()) for phase, g in gpu_df.groupby("phase", sort=False)
}).T.round(3)
display(gpu_summary)
gpu_epochs_df.pivot(index="epoch", columns="rep", values="seconds").round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

cores = cpu_usage_par.filter(like="cpu_").T
im = axes[0].imshow(cores, aspect="auto", cmap="viridis", vmin=0, vmax=100,
                    extent=[cpu_usage_par["t"].min(), cpu_usage_par["t"].max(), len(cores) - 0.5, -0.5])
axes[0].set_title(f"Preprocesamiento con {worker_counts[-1]} procesos")
axes[0].set_xlabel("Tiempo (s)")
axes[0].set_ylabel("Núcleo lógico")
fig.colorbar(im, ax=axes[0], label="Uso del núcleo (%)")

axes[1].plot(gpu_usage["t"], gpu_usage["gpu_util"], color="#55A868")
axes[1].set_title("Uso de la GPU en el entrenamiento (repetición 1)")
axes[1].set_xlabel("Tiempo (s)")
axes[1].set_ylabel("Uso de la GPU (%)")
axes[1].set_ylim(0, 105)

axes[2].plot(gpu_usage["t"], gpu_usage["gpu_mem_mb"], color="#C44E52")
axes[2].set_title("Memoria de la GPU en el entrenamiento (repetición 1)")
axes[2].set_xlabel("Tiempo (s)")
axes[2].set_ylabel("Memoria usada (MB)")

fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "02_resource_usage.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Uso promedio del CPU con {worker_counts[-1]} procesos: {cores.mean().mean():.1f} %")
print(f"Uso promedio de la GPU en el entrenamiento: {gpu_usage['gpu_util'].mean():.1f} %")
print(f"Memoria máxima usada en la GPU: {gpu_usage['gpu_mem_mb'].max():.0f} MB")